문제 1. 계절별 평균 기온
봄(3~5월), 여름(6~8월), 가을(9~11월), 겨울(12월·1월·2월)의 도시별 평균 기온을 (5, 4) 표로 만드세요. 열 순서는 겨울, 봄, 여름, 가을입니다.
추가: 여름과 겨울 평균의 차이가 가장 큰 도시는 어디인가요?
사용할 연산: 슬라이싱, torch.cat, reshape, mean

문제 2. 전달보다 가장 많이 오른 달 · 내린 달
도시마다 기온이 전달보다 가장 많이 오른 달과 그 폭, 가장 많이 내린 달과 그 폭을 구하세요.
(예: 3월 6.0도 → 4월 12.5도이면 4월에 6.5도 오른 것)
사용할 연산: 슬라이싱, 원소별 빼기, max, argmax, argmin

문제 3. 달마다 가장 따뜻한 도시와 가장 추운 도시
1월부터 12월까지 달마다 가장 따뜻한 도시와 가장 추운 도시를 구하세요.
그리고 달마다 가장 따뜻한 도시와 가장 추운 도시의 기온 차이를 구하고, 차이가 큰 계절과 작은 계절을 말해 보세요.
사용할 연산: argmax, argmin, max, min (어느 dim을 접어야 할까요?)

문제 4. 25도 이상인 달의 수와 평균
도시마다 기온이 25도 이상인 달이 몇 개인지 구하세요.
그리고 25도 이상인 달들만의 평균 기온을 도시별로 구하세요.
사용할 연산: 비교 마스크, sum, torch.where

문제 5. 행렬 곱으로 분기 평균 구하기
1분기(1~3월), 2분기(4~6월), 3분기(7~9월), 4분기(10~12월)의 도시별 평균을 행렬 곱 한 번으로 구하세요.
(12, 4) 크기의 행렬을 직접 만들어 기온 @ 행렬 로 계산하고, reshape와 mean으로 구한 결과와 같은지 torch.allclose로 확인하세요.
사용할 연산: torch.zeros, 슬라이싱 대입, @, reshape, mean, allclose


In [5]:
# [2권 Chapter 4. 텐서: 숫자를 담는 그릇 · 4.11 실습: 도시별 기후 패턴 비교하기 · manuscript-vol2/chapters/04-tensors.md:1334]
import torch

도시 = ["가람시", "나래시", "다솜시", "라온시", "마루시"]
# 도시 5곳 × 12개월 월평균 기온(°C). 실제 관측값이 아닌 예제용 숫자입니다.
기온 = torch.tensor(
    [
        [-2.0, 0.5, 6.0, 12.5, 18.0, 22.5, 25.5, 26.0, 21.5, 14.5, 7.0, 0.5],
        [3.5, 5.0, 9.0, 14.0, 18.5, 21.5, 25.0, 26.5, 23.0, 18.0, 12.0, 6.0],
        [-4.0, -1.5, 4.5, 11.5, 17.5, 22.0, 24.5, 25.0, 19.5, 12.5, 5.0, -1.5],
        [6.0, 7.0, 10.0, 14.5, 18.5, 21.5, 26.0, 27.0, 23.5, 19.0, 13.5, 8.5],
        [-1.0, 1.0, 6.5, 13.0, 18.5, 23.0, 25.5, 26.5, 21.0, 14.0, 7.5, 1.0],
    ]
)
# print("모양:", 기온.shape)

# 1) 도시별 연평균과 연교차(가장 더운 달 - 가장 추운 달)
연평균 = 기온.mean(dim=1)
연교차 = 기온.max(dim=1).values - 기온.min(dim=1).values
for 이름, 평균, 교차 in zip(도시, 연평균, 연교차):
    # print(f"{이름}: 연평균 {평균:5.2f}°C, 연교차 {교차:4.1f}°C")
    pass

# 2) 도시마다 가장 더운 달 (argmax는 0부터 세므로 +1)
# print("가장 더운 달:", (기온.argmax(dim=1) + 1).tolist())

print()
# print("가장 추운 달:", (기온.argmin(dim=1) + 1).tolist())
# print(기온.min(dim=1))
# print("도시별 영하인 달의 갯수:", (기온 < 0).sum(dim=1))
# print("연교차가 가장 큰 도시:", 도시[연교차.argmax()])
print()

# 3) 도시마다 정규화: (기온 - 그 도시 평균) / 그 도시 표준편차
평균 = 기온.mean(dim=1, keepdim=True)  # (5, 1)
표준편차 = 기온.std(dim=1, keepdim=True)  # (5, 1)
정규화 = (기온 - 평균) / 표준편차  # (5, 12) - (5, 1) → 브로드캐스팅
# print("정규화 후 도시별 평균:", 정규화.mean(dim=1).abs().max() < 1e-6)

# 월 별 다섯 도시의 평균 기온과 shape
# print("월 별 다섯 도시의 평균 기온:", 기온.mean(dim=0))
# print("월 별 다섯 도시의 평균 기온 (shape):", 기온.mean(dim=0).shape)

# 4) 기후 패턴이 가장 닮은 도시 찾기: 코사인 유사도 = 단위 벡터끼리의 행렬곱
단위 = 정규화 / 정규화.norm(dim=1, keepdim=True)  # keepdim=True로 차원 유지
유사도 = 단위 @ 단위.T  # (5, 12) @ (12, 5) → (5, 5)
유사도.fill_diagonal_(-1.0)  # 자기 자신은 빼기
짝 = 유사도.argmax(dim=1)
for i, j in enumerate(짝.tolist()):
    # print(f"{도시[i]}와 가장 닮은 도시: {도시[j]} (유사도 {유사도[i, j]:.4f})")
    pass

# 5) 도시마다 가장 더운 석 달 (topk는 큰 순서로 주므로 달 순서로 다시 정렬)
더운_석달 = 기온.topk(3, dim=1).indices.sort(dim=1).values + 1
for 이름, 달 in zip(도시, 더운_석달.tolist()):
    # print(f"{이름}의 가장 더운 석 달: {달}월")
    pass

# 6) 난방 지수: 10도보다 추운 만큼을 달마다 더한 값 (예제용 지표)
난방 = torch.where(기온 < 10, 10 - 기온, 0.0)
# print("난방이 필요한 달 수:", (기온 < 10).sum(dim=1).tolist())
# print("난방 지수:", 난방.sum(dim=1).tolist())
# print("clamp로 계산해도 같은가:", torch.equal(난방, (10 - 기온).clamp(min=0)))

# warming_standard to 15
warming_standard = torch.where(기온 < 15, 15 - 기온, 0.0)
# print("난방이 필요한 달 수 (15도 기준):", (기온 < 15).sum(dim=1).tolist())
# print("난방 지수 (15도 기준):", warming_standard.sum(dim=1).tolist())

###
# 문제 1. 계절별 평균 기온
# 봄(3~5월), 여름(6~8월), 가을(9~11월), 겨울(12월·1월·2월)의 도시별 평균 기온을 (5, 4) 표로 만드세요. 열 순서는 겨울, 봄, 여름, 가을입니다.
# 추가: 여름과 겨울 평균의 차이가 가장 큰 도시는 어디인가요?
# 사용할 연산: 슬라이싱, torch.cat, reshape, mean
spring_summer_fall = 기온[:, 2:11].reshape(5, 3, 3).mean(dim=2)  # (5, 3) = 봄, 여름, 가을
winter = torch.cat([기온[:, 11:], 기온[:, :2]], dim=1)  # 12월, 1월, 2월 합치기
winter = winter.mean(dim=1, keepdim=True)  # (5, 1) = 겨울 평균
season = torch.cat([winter, spring_summer_fall], dim=1)  # (5, 4) = 겨울, 봄, 여름, 가을
diff = season[:, 2] - season[:, 0]  # 여름 - 겨울
print("계절별 평균 기온 (열 순서: 겨울, 봄, 여름, 가을):")
print(season)
print("여름과 겨울 평균의 차이가 가장 큰 도시:", 도시[diff.argmax()])

# 문제 2. 전달보다 가장 많이 오른 달 · 내린 달
# 도시마다 기온이 전달보다 가장 많이 오른 달과 그 폭, 가장 많이 내린 달과 그 폭을 구하세요.
# (예: 3월 6.0도 → 4월 12.5도이면 4월에 6.5도 오른 것)
# 사용할 연산: 슬라이싱, 원소별 빼기, max, argmax, argmin
changes = 기온[:, 1:] - 기온[:, :-1]  # 전달 대비 변화량  # (5, 11) = 1월~11월 대비 2월~12월 변화량
print("가장 많이 오른 달과 그 폭:")
print(changes.argmax(dim=1) + 2, changes.max(dim=1).values)  # +2: 변화가 일어난 달 (2월~12월)
print("가장 많이 내린 달과 그 폭:")
print(changes.argmin(dim=1) + 2, changes.min(dim=1).values)  # +2: 변화가 일어난 달 (2월~12월)



계절별 평균 기온 (열 순서: 겨울, 봄, 여름, 가을):
tensor([[-0.3333, 12.1667, 24.6667, 14.3333],
        [ 4.8333, 13.8333, 24.3333, 17.6667],
        [-2.3333, 11.1667, 23.8333, 12.3333],
        [ 7.1667, 14.3333, 24.8333, 18.6667],
        [ 0.3333, 12.6667, 25.0000, 14.1667]])
여름과 겨울 평균의 차이가 가장 큰 도시: 다솜시
가장 많이 오른 달과 그 폭:
tensor([4, 4, 4, 4, 4]) tensor([6.5000, 5.0000, 7.0000, 4.5000, 6.5000])
가장 많이 내린 달과 그 폭:
tensor([11, 11, 11, 11, 10]) tensor([-7.5000, -6.0000, -7.5000, -5.5000, -7.0000])
